# Unit 5 — Unity ML-Agents: SnowballTarget

**Pass criterion:** threshold −100 → **a correctly pushed model is enough.**

The agent (a bear called Julien 🐻) learns to throw snowballs at **moving targets**. +1 reward per hit.

No algorithm code here: the game is a pre-built Unity environment and PPO is provided by ML-Agents. We write the **training config (YAML)** and run the trainer.

## 1. Setup
ML-Agents only supports **Python 3.10**, while Colab ships a newer Python. We therefore create a separate **Python 3.10.12 environment** (`/content/mlenv`) with `uv` and run all ML-Agents commands from there (~3–5 min).

PyTorch is pinned to **2.2.2** (CPU build): recent PyTorch releases switched `torch.onnx.export` to a new exporter that needs `onnxscript` and breaks ML-Agents' checkpoint export.

In [ ]:
!pip install -q uv
!git clone -q --depth 1 https://github.com/Unity-Technologies/ml-agents /content/ml-agents
!uv venv -q -p 3.10.12 /content/mlenv
!uv pip install -q --python /content/mlenv/bin/python -e /content/ml-agents/ml-agents-envs -e /content/ml-agents/ml-agents
# Newest PyTorch needs onnxscript for ONNX export and breaks ML-Agents checkpoints -> pin the version ML-Agents was tested with
!uv pip install -q --python /content/mlenv/bin/python "torch==2.2.2" "numpy<2" --index-url https://download.pytorch.org/whl/cpu --extra-index-url https://pypi.org/simple

In [ ]:
!/content/mlenv/bin/python --version
!/content/mlenv/bin/mlagents-learn --help | head -3

In [ ]:
HF_USERNAME = "Zorlu5454"   # <- your Hugging Face username
ML = "/content/mlenv/bin"       # ML-Agents commands live here

## 2. Hugging Face login (once, at the start)
The model is pushed automatically after training, so we log in first. The cell reads the `HF_TOKEN` Colab secret.

In [ ]:
import os
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")        # Colab secret (🔑 left panel)
    os.environ["HF_TOKEN"] = tok
    login(token=tok, add_to_git_credential=False)
    print("Logged in with Colab secret HF_TOKEN")
except Exception as e:
    print("No Colab secret, falling back to interactive login:", e)
    notebook_login()

## 3. Live training dashboard: TensorBoard
The panel below updates **live** during training (press refresh ⟳ in the top right).
Key chart: **`Environment/Cumulative Reward`** → if it rises, the agent is learning.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/results

---
# SnowballTarget

## A1. Download the environment

In [ ]:
%%capture
!mkdir -p /content/envs
!wget -q "https://github.com/huggingface/Snowball-Target/raw/main/SnowballTarget.zip" -O /content/envs/SnowballTarget.zip
!unzip -o -d /content/envs /content/envs/SnowballTarget.zip
!chmod -R 755 /content/envs/SnowballTarget

## A2. Training config (YAML)
This file tells ML-Agents' built-in PPO **how** to train:
- `max_steps: 200000` → train for 200k steps.
- `learning_rate`, `batch_size`, `buffer_size`, `num_epoch` → PPO update settings.
- `beta` → entropy coefficient (exploration), `epsilon: 0.2` → PPO's **clipping** range.
- `hidden_units: 256, num_layers: 2` → network size.
- `extrinsic` reward → the game's own reward (+1 per hit).

In [ ]:
%%writefile /content/SnowballTarget.yaml
behaviors:
  SnowballTarget:
    trainer_type: ppo
    summary_freq: 10000
    keep_checkpoints: 10
    checkpoint_interval: 50000
    max_steps: 200000
    time_horizon: 64
    threaded: true
    hyperparameters:
      learning_rate: 0.0003
      learning_rate_schedule: linear
      batch_size: 128
      buffer_size: 2048
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
    network_settings:
      normalize: false
      hidden_units: 256
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0

## A3. Training (~20–30 min)
Watch `Mean Reward` in the log: ~2–5 at the start, ~20+ at the end. The same curve appears in TensorBoard above.

In [ ]:
!{ML}/mlagents-learn /content/SnowballTarget.yaml --env=/content/envs/SnowballTarget/SnowballTarget --run-id="SnowballTarget1" --results-dir=/content/results --no-graphics --force

## A4. Push to the Hub

In [ ]:
!{ML}/mlagents-push-to-hf --run-id="SnowballTarget1" --local-dir="/content/results/SnowballTarget1" --repo-id="{HF_USERNAME}/ppo-SnowballTarget" --commit-message="SnowballTarget PPO agent"

## A5. Watch 🎮
1. Open https://huggingface.co/spaces/ThomasSimonini/ML-Agents-SnowballTarget
2. **Step 1:** enter the model id `Zorlu5454/ppo-SnowballTarget`.
3. **Step 2:** select the `.onnx` file → **Watch my agent play** 👀

---
# Check (the same query the certification checker runs)

In [ ]:
from huggingface_hub import HfApi, ModelCard

def check_like_certificate(env_tag, lib_tag, min_result):
    """Same query the course's Check-my-progress Space runs."""
    api = HfApi()
    ids = [m.id for m in api.list_models(author=HF_USERNAME,
                                         filter=["reinforcement-learning", env_tag, lib_tag])]
    if not ids:
        print(f"❌ No model with tags [{env_tag}, {lib_tag}] under {HF_USERNAME} (wait ~1 min and retry)")
        return
    for mid in ids:
        try:
            meta = ModelCard.load(mid).data.to_dict()
            value = str(meta["model-index"][0]["results"][0]["metrics"][0]["value"])
            if "+/-" in value:
                mean, std = (float(x) for x in value.split("+/-"))
            else:
                mean, std = float(value), 0.0
            result = mean - std
            print(f"{'✅' if result >= min_result else '❌'} {mid}: {result:.2f} (needs >= {min_result})")
        except Exception:
            print(f"✅ {mid}: found (no score in card; this unit only needs the upload)")
check_like_certificate("ML-Agents-SnowballTarget", "ml-agents", -100)
